In [ ]:
import pandas as pd

# 1. load dataset from excel file
def load_dataset(path, sheet_name):
    df = pd.read_excel(path, sheet_name=sheet_name)
    # read the intended columns
    df = df[['Question', 'Answer']]
    # remove rows with missing values
    df = df.dropna()
    # reset the row index after dropping rows
    df = df.reset_index(drop=True)
    return df

df = load_dataset('C:\\Users\\diya0\\Desktop\\sap gen ai\\poornima_bot\\pu_chatbot.xlsx', 'Sheet1')
print(df.head())



                                 Question  \
0  I want to know more about the training   
1    I want to know more about the server   
2                  How much does it cost?   
3          What are your payment options?   
4            how do you share the content   

                                              Answer  
0  Please let me know which course are you lookin...  
1  Yes, we provide on-demand server access where ...  
2  It depends on which module you want to take tr...  
3  You can make an online transfer to our bank a/...  
4  We will share the private blog access with you...  


In [9]:
# preprocessing: clean the data from the dataset
import nltk
import pandas as pd
from nltk.tokenize import WhitespaceTokenizer

# download nltk resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# cleanse the data
def clean_data(data_frame):
    word_tokenizer = WhitespaceTokenizer()
    stop_words = set(nltk.corpus.stopwords.words('english'))
    lemmatizer = nltk.stem.WordNetLemmatizer()

    cleaned_data = []
    for _, row in data_frame.iterrows():
        question = row['Question']
        answer = row['Answer']

        # tokenize and clean question
        question_tokens = word_tokenizer.tokenize(question)
        question_tokens = [word.lower() for word in question_tokens if word.isalpha()]
        question_tokens = [lemmatizer.lemmatize(word) for word in question_tokens if word not in stop_words]
        cleaned_question = ' '.join(question_tokens)

        # tokenize and clean answer
        answer_tokens = word_tokenizer.tokenize(answer)
        answer_tokens = [word.lower() for word in answer_tokens if word.isalpha()]
        answer_tokens = [lemmatizer.lemmatize(word) for word in answer_tokens if word not in stop_words]
        cleaned_answer = ' '.join(answer_tokens)

        cleaned_data.append({"Question": cleaned_question})

    return pd.DataFrame(cleaned_data)

cleaned_data = clean_data(df)
print(cleaned_data.head())

             Question
0  want know training
1    want know server
2                much
3             payment
4       share content


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\diya0\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\diya0\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\diya0\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [11]:
##vectorization the data of anubhav chatbot
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize

def process_questions(user_input, data_frame):
    #for simplicity, we will just echo back the question
    print(f"YOU ASKED: {user_input}")
    #get all questions in list
    questions = data_frame['Question'].tolist()
    print(f"ALL QUESTIONS: {questions}")
    ##add users question as first line
    questions.append(user_input)
    ##calculate tf-idf for all data
    vectorizer = TfidfVectorizer(tokenizer=word_tokenize, stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(questions)
    #print the matrix
    print(tfidf_matrix.toarray())
    #cosine similarity
    cosine_sim = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])
    print(f"COSINE SIMILARITY: {cosine_sim}")
    
    #index of the line no. of record with most similar
    most_similar_index = cosine_sim.argmax()
    print(f"MOST SIMILAR INDEX: {most_similar_index}")

    #remove question from dataset
    questions.pop()

    #return the answers
    answer = data_frame.iloc[most_similar_index]['Answer']
    print(f"ANSWER: {answer}")

    #return the matrix
    return answer

print(process_questions("is this training real time", df))


YOU ASKED: is this training real time
ALL QUESTIONS: ['I want to know more about the training', 'I want to know more about the server', 'How much does it cost?', 'What are your payment options?', 'how do you share the content', 'Where are you located?', 'How may I contact you?', 'Are you a chatbot?', 'I need help', 'Are you a real person?', 'Can I ask you few questions?', 'You have not answered my question', 'There is a problem it is not working', 'I want to know about UI5 & Fiori training', 'i want to know course details', 'I have already paid', 'who is the trainer', 'I want to know about ABAP on HANA training', 'need course details of abap on cloud', 'need course details of Fiori Security and Launchpad', 'which course is suitable for me', 'is anubhav obeory the trainer', 'How do I register for the classes?', 'Is training Real time?', 'Will it be 1:1 Training?', 'What are all be part of training package?', 'do you provide certification material', 'How much will be the duration of the 

c:\Users\diya0\Desktop\sap gen ai\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
